## MMTHE01 - Masters Thesis

### B. Thesis - Feature Engineering - Imputing missing fields
* Data Encoding
* Feature Scaling

#### Importing the libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
import pickle

from sklearn.preprocessing import LabelEncoder

In [2]:
# Check your current working directory
cwd = os.getcwd()

# Define your relative path
relative_path = r"Analysis"  # adjust this relative to cwd

# Build the full path
full_path = os.path.join(cwd, relative_path)

# Check if it exists before changing
if os.path.exists(full_path):
    os.chdir(full_path)
    print("Changed directory to:", full_path)
else:
    print("Folder does not exist:", full_path)

Changed directory to: C:\Users\eaber\Documents\Masters Thesis Final\Analysis


#### Importing the dataset

In [3]:
### Mark categorical fields to ensure they are appropriately delt with
categorical_columns_all = ['term', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'verification_status', 'issue_d', 'purpose', 'title',
                       'zip_code', 'addr_state', 'earliest_cr_line', 'initial_list_status']

In [4]:
dataset = pd.read_csv('final_lending_club.csv', dtype={col: 'category' for col in categorical_columns_all}, low_memory=False)

In [5]:
### Import features from step on
intermediate_output_path = os.path.join(r'intermediate_output', 'final_features_to_stay.csv')
features_to_stay =  pd.read_csv(intermediate_output_path)

In [6]:
features_to_stay_list = features_to_stay["features_to_stay"].tolist()

### Dataset after Step 1

In [7]:
dataset_final = dataset[features_to_stay_list].copy()

In [8]:
dataset_final.head()

,loan_amnt,term,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,purpose,...,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,year_issued,default_status
0,500,36 months,A5,us army,2 years,RENT,33000.0,Not Verified,Sep-2007,other,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,2007,0
1,6000,36 months,D5,SUNY- ESF,2 years,RENT,12000.0,Not Verified,Jul-2007,credit_card,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007,0
2,14700,36 months,E4,Labcorp,10+ years,MORTGAGE,59000.0,Not Verified,Nov-2007,credit_card,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,2007,0
3,5000,36 months,E1,Ray Catena of Union LLC,3 years,MORTGAGE,98000.0,Not Verified,Nov-2007,credit_card,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,2007,0
4,3750,36 months,F3,Private family,5 years,MORTGAGE,7280.0,Not Verified,Aug-2007,moving,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,2007,0


In [9]:
dataset_final.head(100).to_csv('intermediate_output/dataset_final.csv', index=False)

In [10]:
def top_missing_cols_perc(df):
    """
    returns missing columns in dataframe with missing values percent > thresh
    if n=None. It will gave whole dataframe with missing values percent > thresh
    """
    
    dff = (df.isnull().sum()/df.shape[0])*100
    dff = dff.reset_index()
    dff.columns = ['col','missing_percent']
    dff = dff.sort_values(by=['missing_percent'],ascending=True).reset_index(drop=True)
    return dff

In [11]:
missing_perc_matrix = top_missing_cols_perc(dataset_final)

In [12]:
missing_perc_matrix.to_csv('intermediate_output/missing_perc_matrix_values2.csv', index=False)

### Step 2: Use SimpleImputer to impute missing feature values

#### 2.1 Impute missing values for categorical variables

##### 2.1.1 Get the cat features

In [13]:
columns_with_missing = dataset_final.columns[dataset_final.isnull().any()].to_list()
len(columns_with_missing)

47

In [14]:
dict_ = {}
for column in columns_with_missing:
    dict_[column] = dataset_final.dtypes[column]

In [15]:
#### Get caregorical feature columns with missing data plus the columns without missing data
categorical_columns = [column for column in columns_with_missing if dataset_final[column].dtype == 'category']
len(categorical_columns)

4

In [16]:
categorical_columns_dataset = dataset_final[categorical_columns].copy()
categorical_columns_dataset.head()

,emp_title,emp_length,title,earliest_cr_line
0,us army,2 years,Paying bills.,Apr-1990
1,SUNY- ESF,2 years,Credit Card,Dec-2004
2,Labcorp,10+ years,consolidation,Dec-1987
3,Ray Catena of Union LLC,3 years,Refinancing Debt,Jan-2000
4,Private family,5 years,Recent grad needs to move to Chicago!,Jan-1998


In [17]:
cat_missing_check = categorical_columns_dataset.columns[categorical_columns_dataset.isnull().any()].to_list()
len(cat_missing_check)

4

##### 2.1.2 Impute missing categorical values

In [18]:
from sklearn.impute import SimpleImputer

In [19]:
cat_imputer = SimpleImputer(strategy='most_frequent')

In [20]:
imputed_cat_data = cat_imputer.fit_transform(categorical_columns_dataset)

In [21]:
imputed_cat_df = pd.DataFrame(imputed_cat_data, columns=categorical_columns_dataset.columns)

In [22]:
imputed_cat_df.head()

,emp_title,emp_length,title,earliest_cr_line
0,us army,2 years,Paying bills.,Apr-1990
1,SUNY- ESF,2 years,Credit Card,Dec-2004
2,Labcorp,10+ years,consolidation,Dec-1987
3,Ray Catena of Union LLC,3 years,Refinancing Debt,Jan-2000
4,Private family,5 years,Recent grad needs to move to Chicago!,Jan-1998


In [23]:
cat_missing_check = imputed_cat_df.columns[imputed_cat_df.isnull().any()].to_list()
cat_missing_check

[]

#### 2.2 Impute missing values for numeric variable

##### 2.2.1 Get the numeric features

In [24]:
columns_with_missing = dataset_final.columns[dataset_final.isnull().any()].to_list()
len(columns_with_missing)

47

In [25]:
dict_ = {}
for column in columns_with_missing:
    dict_[column] = dataset_final.dtypes[column]

In [26]:
#### Get numeric feature columns with missing data plus the columns without missing data
numeric_columns = [column for column in columns_with_missing if dataset_final[column].dtype == 'float64']
len(numeric_columns)

43

In [27]:
numeric_columns_dataset = dataset_final[numeric_columns].copy()
numeric_columns_dataset.head()

,dti,delinq_2yrs,inq_last_6mths,open_acc,pub_rec,revol_util,total_acc,collections_12_mths_ex_med,tot_coll_amt,total_rev_hi_lim,...,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit
0,2.15,0.0,10.0,6.0,0.0,18.6,6.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
1,4.40,0.0,9.0,15.0,0.0,49.3,16.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,23.86,0.0,4.0,18.0,0.0,60.2,50.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
3,12.53,0.0,1.0,7.0,3.0,97.1,12.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
4,17.31,0.0,1.0,1.0,0.0,NaN,4.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN


In [28]:
num_missing_check = numeric_columns_dataset.columns[numeric_columns_dataset.isnull().any()].to_list()
len(num_missing_check)

43

##### 2.2.2 Impute missing numeric values

In [29]:
num_imputer = SimpleImputer(strategy='mean')

In [30]:
imputed_num_data = num_imputer.fit_transform(numeric_columns_dataset)

In [31]:
imputed_num_df = pd.DataFrame(imputed_num_data, columns=numeric_columns_dataset.columns)

In [32]:
imputed_num_df.head()

,dti,delinq_2yrs,inq_last_6mths,open_acc,pub_rec,revol_util,total_acc,collections_12_mths_ex_med,tot_coll_amt,total_rev_hi_lim,...,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit
0,2.15,0.0,10.0,6.0,0.0,18.600000,6.0,0.000000,222.17346,34606.178028,...,0.083188,2.077516,94.11971,42.361183,0.127908,0.000000,178248.102764,51080.557557,23226.657827,43762.092547
1,4.40,0.0,9.0,15.0,0.0,49.300000,16.0,0.018288,222.17346,34606.178028,...,0.083188,2.077516,94.11971,42.361183,0.127908,0.046793,178248.102764,51080.557557,23226.657827,43762.092547
2,23.86,0.0,4.0,18.0,0.0,60.200000,50.0,0.000000,222.17346,34606.178028,...,0.083188,2.077516,94.11971,42.361183,0.127908,0.000000,178248.102764,51080.557557,23226.657827,43762.092547
3,12.53,0.0,1.0,7.0,3.0,97.100000,12.0,0.000000,222.17346,34606.178028,...,0.083188,2.077516,94.11971,42.361183,0.127908,0.000000,178248.102764,51080.557557,23226.657827,43762.092547
4,17.31,0.0,1.0,1.0,0.0,50.318895,4.0,0.018288,222.17346,34606.178028,...,0.083188,2.077516,94.11971,42.361183,0.127908,0.000000,178248.102764,51080.557557,23226.657827,43762.092547


In [33]:
num_missing_check = imputed_num_df.columns[imputed_num_df.isnull().any()].to_list()
num_missing_check

[]

### Step 3: Combining the data, feature engineering and encoding categorical data using the label encoder

#### 3.1 Combine the data

In [34]:
full_columns = dataset_final.columns[dataset_final.notna().all()].to_list()

In [35]:
dataset_full_features = dataset_final[full_columns]

In [36]:
dataset_full_features.head(1)

,loan_amnt,term,sub_grade,home_ownership,annual_inc,verification_status,issue_d,purpose,zip_code,addr_state,revol_bal,initial_list_status,policy_code,year_issued,default_status
0,500,36 months,A5,RENT,33000.0,Not Verified,Sep-2007,other,664xx,KS,3461,f,1,2007,0


In [37]:
dataset_j1 = dataset_full_features.merge(imputed_cat_df, on= imputed_cat_df.index ,how='inner' )
dataset_j1.drop('key_0', axis=1, inplace=True)

In [38]:
dataset_j1.head(1)

,loan_amnt,term,sub_grade,home_ownership,annual_inc,verification_status,issue_d,purpose,zip_code,addr_state,revol_bal,initial_list_status,policy_code,year_issued,default_status,emp_title,emp_length,title,earliest_cr_line
0,500,36 months,A5,RENT,33000.0,Not Verified,Sep-2007,other,664xx,KS,3461,f,1,2007,0,us army,2 years,Paying bills.,Apr-1990


In [39]:
full_dataset = dataset_j1.merge(imputed_num_df, on= imputed_num_df.index ,how='inner' )
full_dataset.drop('key_0', axis=1, inplace=True)

In [40]:
full_dataset.head(1)

,loan_amnt,term,sub_grade,home_ownership,annual_inc,verification_status,issue_d,purpose,zip_code,addr_state,...,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit
0,500,36 months,A5,RENT,33000.0,Not Verified,Sep-2007,other,664xx,KS,...,0.083188,2.077516,94.11971,42.361183,0.127908,0.0,178248.102764,51080.557557,23226.657827,43762.092547


In [41]:
full_dataset.shape

(226067, 62)

In [42]:
full_data_missing_check = full_dataset.columns[full_dataset.isnull().any()].to_list()
full_data_missing_check

[]

#### 3.2: Dealing with Date Features

In [43]:
full_dataset[['issue_d','earliest_cr_line']].head()

,issue_d,earliest_cr_line
0,Sep-2007,Apr-1990
1,Jul-2007,Dec-2004
2,Nov-2007,Dec-1987
3,Nov-2007,Jan-2000
4,Aug-2007,Jan-1998


In [44]:
full_dataset['issue_year'] = full_dataset['issue_d'].apply(lambda x: x.split('-')[1])
full_dataset['issue_month'] = full_dataset['issue_d'].apply(lambda x: x.split('-')[0])

In [45]:
full_dataset = full_dataset.drop('year_issued', axis=1)

In [46]:
full_dataset['earliest_cr_line_year'] = full_dataset['earliest_cr_line'].apply(lambda x: str(x).split('-')[1])
full_dataset['earliest_cr_line_month'] = full_dataset['earliest_cr_line'].apply(lambda x: str(x).split('-')[0])

In [47]:
full_dataset['term_in_months'] = full_dataset['term'].apply(lambda x: x.split(' ')[1])

In [48]:
full_dataset[['issue_d','earliest_cr_line','issue_year','issue_month','earliest_cr_line_year','earliest_cr_line_month','term_in_months']].head()

,issue_d,earliest_cr_line,issue_year,issue_month,earliest_cr_line_year,earliest_cr_line_month,term_in_months
0,Sep-2007,Apr-1990,2007,Sep,1990,Apr,36
1,Jul-2007,Dec-2004,2007,Jul,2004,Dec,36
2,Nov-2007,Dec-1987,2007,Nov,1987,Dec,36
3,Nov-2007,Jan-2000,2007,Nov,2000,Jan,36
4,Aug-2007,Jan-1998,2007,Aug,1998,Jan,36


In [49]:
### Drop the raw date features
full_dataset = full_dataset.drop(['issue_d','earliest_cr_line','term'],axis=1)

#### 3.4 Export data Unencoded data

In [50]:
# Save all datasets in one file
with open("saved_data/final_dataset_unencoded.pkl", "wb") as f:
    pickle.dump((full_dataset), f)

In [51]:
full_dataset.shape

(226067, 63)

#### 3.3 Encode categorical data

##### 3.2.1 Understanding the categorical variables which need encoding

In [52]:
#list of categorical features which require encoding 
categorical_cols = ['sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'title', 'issue_month', 
                    'earliest_cr_line_month', 'zip_code', 'addr_state', 'initial_list_status']

In [53]:
# Count unique values in each categorical column
category_counts = full_dataset[categorical_cols].nunique()

print(category_counts)

sub_grade                    35
emp_title                 81480
emp_length                   11
home_ownership                6
verification_status           3
purpose                      14
title                      8758
issue_month                  12
earliest_cr_line_month       12
zip_code                    898
addr_state                   51
initial_list_status           2
dtype: int64


In [54]:
### Encoding methods to use:
#1. Use Label encoding - sub_grade, emp_title, title, zip_code, addr_state (because of high cardinality)
#2. Using One Hot Encoding (OHE) - emp_length, home_ownership, verification_status, purpose, issue_month, earliest_cr_line_month, initial_list_status (low cardinality so it is efficient to use OHE)

##### 3.2.2 Label Encoding

In [55]:
def label_encode_keep_nan(series):
    """
    Label encodes a pandas Series while leaving NaNs untouched.
    Returns the encoded series and the fitted encoder.
    """
    # Keep mask of missing values
    mask = series.isna()

    # Apply LabelEncoder to non-NaN values only
    le = LabelEncoder()
    encoded = le.fit_transform(series[~mask])

    # Create a full-length array with NaNs
    full_encoded = pd.Series(np.nan, index=series.index)
    full_encoded[~mask] = encoded

    return full_encoded, le

In [56]:
full_dataset['sub_grade'], sub_grade_encoder = label_encode_keep_nan(full_dataset['sub_grade'])
full_dataset['emp_title'], emp_title_encoder = label_encode_keep_nan(full_dataset['emp_title'])
full_dataset['title'], title_encoder = label_encode_keep_nan(full_dataset['title'])
full_dataset['zip_code'], zip_code_encoder = label_encode_keep_nan(full_dataset['zip_code'])
full_dataset['addr_state'], addr_state_encoder = label_encode_keep_nan(full_dataset['addr_state'])

##### 3.2.3 One Hot Encoding

In [57]:
from sklearn.preprocessing import OneHotEncoder

In [58]:
cols_for_ohe = ['emp_length', 'home_ownership', 'verification_status', 'purpose', 'issue_month', 'earliest_cr_line_month', 'initial_list_status']

In [59]:
# Initialize OHE encoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

In [60]:
# Fit and transform
encoded_array = encoder.fit_transform(full_dataset[cols_for_ohe])

In [61]:
# Create new DataFrame with encoded column names
encoded_df = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(cols_for_ohe))

In [62]:
# Combine with the rest of the data
full_dataset_final = pd.concat([full_dataset.drop(columns=cols_for_ohe), encoded_df], axis=1)

In [63]:
full_dataset_final.head()

,loan_amnt,sub_grade,annual_inc,zip_code,addr_state,revol_bal,policy_code,default_status,emp_title,title,...,earliest_cr_line_month_Jan,earliest_cr_line_month_Jul,earliest_cr_line_month_Jun,earliest_cr_line_month_Mar,earliest_cr_line_month_May,earliest_cr_line_month_Nov,earliest_cr_line_month_Oct,earliest_cr_line_month_Sep,initial_list_status_f,initial_list_status_w
0,500,4.0,33000.0,603.0,16.0,3461,1,0,80998.0,5404.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,6000,19.0,12000.0,122.0,34.0,5251,1,0,54628.0,1527.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,14700,23.0,59000.0,257.0,27.0,14301,1,0,34339.0,7118.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,5000,20.0,98000.0,93.0,34.0,9568,1,0,51372.0,5870.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,3750,27.0,7280.0,550.0,14.0,0,1,0,48332.0,5780.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [64]:
full_dataset_final.groupby('default_status').count()

,loan_amnt,sub_grade,annual_inc,zip_code,addr_state,revol_bal,policy_code,emp_title,title,dti,...,earliest_cr_line_month_Jan,earliest_cr_line_month_Jul,earliest_cr_line_month_Jun,earliest_cr_line_month_Mar,earliest_cr_line_month_May,earliest_cr_line_month_Nov,earliest_cr_line_month_Oct,earliest_cr_line_month_Sep,initial_list_status_f,initial_list_status_w
default_status,,,,,,,,,,,,,,,,,,,,,
0,199898,199898,199898,199898,199898,199898,199898,199898,199898,199898,...,199898,199898,199898,199898,199898,199898,199898,199898,199898,199898
1,26169,26169,26169,26169,26169,26169,26169,26169,26169,26169,...,26169,26169,26169,26169,26169,26169,26169,26169,26169,26169


In [65]:
full_dataset_final['default_status'].value_counts(normalize=True) * 100

default_status
0    88.424228
1    11.575772
Name: proportion, dtype: float64

#### 3.3 Export encoded data

In [66]:
# Save datasets in one file
with open("saved_data/final_dataset_encoded.pkl", "wb") as f:
    pickle.dump((full_dataset_final), f)

In [67]:
full_dataset_final.shape

(226067, 116)